In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model='qwen/qwen3.8-27b', max_tokens=200)
embedding_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001', output_dimensionality=100)

In [12]:
# Indexing
video_id = "z_MoCiIfEPI"

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    transcript_list = transcript_list.to_raw_data()

    # flatten it to plain text
    transcript = " ".join(chunk['text'] for chunk in transcript_list)
    
except TranscriptsDisabled:
    print("No captions available for this video.")

In [13]:
transcript

'How do you decide that? How do you find a non-consensus bet? >> Uh good question. I I like My own experience of this was this was one thing that just no one ever taught me. And you know, like very little of my experience in school was figuring out what questions to ask. Um you got to like answer other people\'s questions and then at some point, I think like when you become a PhD student or whatever, you get to start asking more questions and then eventually do more and more of that, but um I I think we should focus much more on teaching people teaching very young people to figure out what questions to ask and what things are interesting to explore that we don\'t already know. And I think getting good at that skill was the best way that I figured out the sort of non-consensus bets to take. >> Just figuring out how to ask good questions. >> Figuring out how to ask good questions. >> Can you learn that? >> I think so. Yeah. Uh I mean like all of these things, maybe not everybody\'s going

In [14]:
# Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [15]:
chunks

[Document(metadata={}, page_content="How do you decide that? How do you find a non-consensus bet? >> Uh good question. I I like My own experience of this was this was one thing that just no one ever taught me. And you know, like very little of my experience in school was figuring out what questions to ask. Um you got to like answer other people's questions and then at some point, I think like when you become a PhD student or whatever, you get to start asking more questions and then eventually do more and more of that, but um I I think we should focus much more on teaching people teaching very young people to figure out what questions to ask and what things are interesting to explore that we don't already know. And I think getting good at that skill was the best way that I figured out the sort of non-consensus bets to take. >> Just figuring out how to ask good questions. >> Figuring out how to ask good questions. >> Can you learn that? >> I think so. Yeah. Uh I mean like all of these th

In [16]:
# Embedding Generation and Storing in vectoreDB
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [18]:
vectorstore.index_to_docstore_id

{0: 'a67da188-9d6d-400e-bab2-f7b32085f121',
 1: '4597e0bd-ed9d-4b3c-a1b2-6b90603a36de',
 2: '3e4dc80f-3b3e-412e-a2e5-520c26efedc2',
 3: '92dee161-7365-4ef8-bad3-5844532db38b'}

In [20]:
vectorstore.get_by_ids(["4597e0bd-ed9d-4b3c-a1b2-6b90603a36de"])

[Document(id='4597e0bd-ed9d-4b3c-a1b2-6b90603a36de', metadata={}, page_content="Just figuring out how to ask good questions. >> Figuring out how to ask good questions. >> Can you learn that? >> I think so. Yeah. Uh I mean like all of these things, maybe not everybody's going to have equal aptitude for everything, but I think this is one that a lot of people seem to be able to learn. >> You know, there's I forget which which culture which country has this rule where the kid when when he comes back from school, the mother asks the kid that what's the question that you asked today. >> Really? >> Yeah. >> Do you know what culture that's very interesting. Do you know what culture? >> I think it's I think it's Israel or something like someone in Israel. I I'm not sure, but there is some culture where mothers just ask the kids. They don't ask about the highlight. They ask about the what's one question did you ask today? >> That's cool. >> Is that a good habit that we should >> I'm going to st

In [23]:
# Retrieval
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":2})

In [24]:
retriever.invoke("what is this video about?")

[Document(id='a67da188-9d6d-400e-bab2-f7b32085f121', metadata={}, page_content="How do you decide that? How do you find a non-consensus bet? >> Uh good question. I I like My own experience of this was this was one thing that just no one ever taught me. And you know, like very little of my experience in school was figuring out what questions to ask. Um you got to like answer other people's questions and then at some point, I think like when you become a PhD student or whatever, you get to start asking more questions and then eventually do more and more of that, but um I I think we should focus much more on teaching people teaching very young people to figure out what questions to ask and what things are interesting to explore that we don't already know. And I think getting good at that skill was the best way that I figured out the sort of non-consensus bets to take. >> Just figuring out how to ask good questions. >> Figuring out how to ask good questions. >> Can you learn that? >> I thi

In [25]:
prompt = PromptTemplate(
    template="""
        You're a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.
        
        {context}
        Question: {question}
        """,
        input_variables=["context", "question"]
)

In [26]:
question = "what topic is discussed in the video"
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='a67da188-9d6d-400e-bab2-f7b32085f121', metadata={}, page_content="How do you decide that? How do you find a non-consensus bet? >> Uh good question. I I like My own experience of this was this was one thing that just no one ever taught me. And you know, like very little of my experience in school was figuring out what questions to ask. Um you got to like answer other people's questions and then at some point, I think like when you become a PhD student or whatever, you get to start asking more questions and then eventually do more and more of that, but um I I think we should focus much more on teaching people teaching very young people to figure out what questions to ask and what things are interesting to explore that we don't already know. And I think getting good at that skill was the best way that I figured out the sort of non-consensus bets to take. >> Just figuring out how to ask good questions. >> Figuring out how to ask good questions. >> Can you learn that? >> I thi

In [28]:
context_text = " ".join(doc.page_content for doc in retrieved_docs)
context_text

"How do you decide that? How do you find a non-consensus bet? >> Uh good question. I I like My own experience of this was this was one thing that just no one ever taught me. And you know, like very little of my experience in school was figuring out what questions to ask. Um you got to like answer other people's questions and then at some point, I think like when you become a PhD student or whatever, you get to start asking more questions and then eventually do more and more of that, but um I I think we should focus much more on teaching people teaching very young people to figure out what questions to ask and what things are interesting to explore that we don't already know. And I think getting good at that skill was the best way that I figured out the sort of non-consensus bets to take. >> Just figuring out how to ask good questions. >> Figuring out how to ask good questions. >> Can you learn that? >> I think so. Yeah. Uh I mean like all of these things, maybe not everybody's going to

In [29]:
final_prompt = prompt.invoke({'context': context_text, 'question': question})

In [30]:
final_prompt

StringPromptValue(text="\n        You're a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n\n        How do you decide that? How do you find a non-consensus bet? >> Uh good question. I I like My own experience of this was this was one thing that just no one ever taught me. And you know, like very little of my experience in school was figuring out what questions to ask. Um you got to like answer other people's questions and then at some point, I think like when you become a PhD student or whatever, you get to start asking more questions and then eventually do more and more of that, but um I I think we should focus much more on teaching people teaching very young people to figure out what questions to ask and what things are interesting to explore that we don't already know. And I think getting good at that skill was the best way that I figured out the sort of non-consensus bets to take. >> J

In [32]:
# Generation
answer = llm.invoke(final_prompt)
answer.content

'Based on the provided transcript, the topic discussed is how to identify "non-consensus bets" or interesting things to explore, and the speaker argues that this is achieved by learning the skill of figuring out what questions to ask. The conversation also touches on the idea that education should focus more on teaching young people to ask questions rather than just answering them, and it mentions a cultural practice (possibly in Israel) where parents ask their children what questions they asked at school.'

Building a Chain

In [33]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [34]:
parser = StrOutputParser()

In [35]:
def format_docs(retrieved_docs):
    context_text = "/n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'question': RunnablePassthrough(),
    'context': retriever | RunnableLambda(format_docs)
})

In [37]:
parallel_chain.invoke("what is the topic discussed")

{'question': 'what is the topic discussed',
 'context': "How do you decide that? How do you find a non-consensus bet? >> Uh good question. I I like My own experience of this was this was one thing that just no one ever taught me. And you know, like very little of my experience in school was figuring out what questions to ask. Um you got to like answer other people's questions and then at some point, I think like when you become a PhD student or whatever, you get to start asking more questions and then eventually do more and more of that, but um I I think we should focus much more on teaching people teaching very young people to figure out what questions to ask and what things are interesting to explore that we don't already know. And I think getting good at that skill was the best way that I figured out the sort of non-consensus bets to take. >> Just figuring out how to ask good questions. >> Figuring out how to ask good questions. >> Can you learn that? >> I think so. Yeah. Uh I mean 

In [40]:
final_chain = parallel_chain | prompt | llm | parser
answer = final_chain.invoke("what is the topic discussed")
answer

"Based on the transcript, the topic discussed is **how to find non-consensus bets**, which the speaker identifies as being learned by developing the skill of **asking good questions**. The conversation also touches on the importance of teaching young people to figure out what questions to ask and what things are interesting to explore, rather than just answering others' questions. Additionally, they discuss a cultural habit (possibly from Israel) where parents ask children what question they asked in school that day."